# QC: `commission_monthly` по месяцам (дыра августа?)

Быстрая проверка гипотезы: маленький `fin_result` за август из‑за нулевых/отсутствующих месячных комиссий (MPOS_RENT).

## Как запускать
1. **Предпочтительно** — в том же kernel, где уже есть `final_df` / `final_df_period_df` / `final_df_by_month`.
2. Если переменной нет — тетрадка подхватит CSV (путь в следующей ячейке).

Смотри строку **2026-08** vs **2026-07**: `sum_commission_monthly`, `% нулей`, `share_nonzero`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# fallback, если final_df ещё не в памяти этого kernel
CSV_FALLBACK = Path('/home/jovyan/documents/Equaring/Data/final_df_period_2026_01_2026_08_final_script_2.csv')

FOCUS_MONTHS = ['2026-07', '2026-08']


In [ ]:
def resolve_final_df():
    """Take final_df from memory (same kernel) or CSV fallback."""
    candidates = []
    for name in ['final_df_period_df', 'final_df', 'final_df_period']:
        if name in globals() and globals()[name] is not None:
            obj = globals()[name]
            if isinstance(obj, pd.DataFrame) and len(obj):
                candidates.append((name, obj))
    if 'final_df_by_month' in globals() and isinstance(globals().get('final_df_by_month'), dict):
        parts = [df for df in final_df_by_month.values() if isinstance(df, pd.DataFrame) and len(df)]
        if parts:
            candidates.append(('final_df_by_month(concat)', pd.concat(parts, ignore_index=True)))

    if candidates:
        name, df = candidates[0]
        print(f'OK: using in-memory `{name}` rows={len(df):,}')
        return df.copy()

    if CSV_FALLBACK.exists():
        df = pd.read_csv(CSV_FALLBACK, dtype={'inn': 'string', 'agr_id': 'string', 'n_agr': 'string'})
        print(f'OK: loaded CSV fallback rows={len(df):,} | {CSV_FALLBACK}')
        return df

    raise RuntimeError(
        'final_df not found in memory and CSV missing.\n'
        f'Expected one of: final_df_period_df / final_df / final_df_by_month\n'
        f'or file: {CSV_FALLBACK}'
    )


df = resolve_final_df()
print('columns has commission_monthly =', 'commission_monthly' in df.columns)
print('report_month values =', sorted(df['report_month'].astype(str).unique().tolist()) if 'report_month' in df.columns else None)
display(df.head(3))


## Главная таблица: комиссии / ЧОД / финрез по месяцам

Если за **2026-08** `sum_commission_monthly ≈ 0` (или на порядок меньше июля) при живом `commission_from_ops` — гипотеза подтверждается.


In [ ]:
need = ['report_month', 'commission_monthly']
missing = [c for c in need if c not in df.columns]
if missing:
    raise RuntimeError(f'missing columns: {missing}')

work = df.copy()
work['report_month'] = work['report_month'].astype(str)
for c in ['commission_monthly', 'commission_from_ops', 'commission_total', 'int_component', 'chod', 'aur', 'amortization', 'fin_result']:
    if c in work.columns:
        work[c] = pd.to_numeric(work[c], errors='coerce')

cm = pd.to_numeric(work['commission_monthly'], errors='coerce')
work['_cm'] = cm
work['_cm_nonzero'] = cm.fillna(0).ne(0).astype(int)

agg_map = {
    'rows': ('_cm', 'size'),
    'sum_commission_monthly': ('_cm', 'sum'),
    'nonzero_n': ('_cm_nonzero', 'sum'),
}
for c, alias in [
    ('commission_from_ops', 'sum_commission_from_ops'),
    ('commission_total', 'sum_commission_total'),
    ('int_component', 'sum_int_component'),
    ('chod', 'sum_chod'),
    ('aur', 'sum_aur'),
    ('amortization', 'sum_amortization'),
    ('fin_result', 'sum_fin_result'),
]:
    if c in work.columns:
        agg_map[alias] = (c, 'sum')

monthly = (
    work.groupby('report_month', as_index=False)
        .agg(**agg_map)
        .sort_values('report_month')
        .reset_index(drop=True)
)
monthly['zero_or_na_n'] = monthly['rows'] - monthly['nonzero_n']
monthly['share_nonzero_pct'] = (100.0 * monthly['nonzero_n'] / monthly['rows']).round(2)
monthly['share_zero_pct'] = (100.0 * monthly['zero_or_na_n'] / monthly['rows']).round(2)

print('=== commission_monthly by month ===')
display(monthly)

# July vs August spotlight
spot = monthly.loc[monthly['report_month'].isin(FOCUS_MONTHS)].copy()
print('\n=== FOCUS July vs August ===')
display(spot)

if set(FOCUS_MONTHS).issubset(set(monthly['report_month'])):
    jul = monthly.loc[monthly['report_month'] == '2026-07'].iloc[0]
    aug = monthly.loc[monthly['report_month'] == '2026-08'].iloc[0]
    print('\n=== Verdict hint ===')
    print('July sum_commission_monthly =', f"{jul['sum_commission_monthly']:,.2f}")
    print('August sum_commission_monthly =', f"{aug['sum_commission_monthly']:,.2f}")
    if jul['sum_commission_monthly'] and abs(aug['sum_commission_monthly']) < 0.05 * abs(jul['sum_commission_monthly']):
        print('→ YES: August monthly commission looks MISSING/near-zero vs July. This can explain small fin_result.')
    elif aug['sum_commission_monthly'] < 0.5 * jul['sum_commission_monthly']:
        print('→ PARTIAL: August monthly commission much lower than July — likely contributes to small fin_result.')
    else:
        print('→ Monthly commission not the main hole (August comparable to July). Look at commission_from_ops / IRF / chod.')
    if 'sum_chod' in monthly.columns and 'sum_fin_result' in monthly.columns:
        print('July chod=', f"{jul.get('sum_chod', float('nan')):,.2f}", 'fin_result=', f"{jul.get('sum_fin_result', float('nan')):,.2f}")
        print('August chod=', f"{aug.get('sum_chod', float('nan')):,.2f}", 'fin_result=', f"{aug.get('sum_fin_result', float('nan')):,.2f}")


## Дополнительно: источник MPOS (если колонка есть)

Если в `final_df` есть `commission_monthly_source` — смотри долю `mpos_rent_missing` за август.


In [ ]:
if 'commission_monthly_source' in work.columns:
    src = (
        work.groupby(['report_month', 'commission_monthly_source'], dropna=False)
            .size()
            .rename('rows')
            .reset_index()
            .sort_values(['report_month', 'rows'], ascending=[True, False])
    )
    print('=== commission_monthly_source by month ===')
    display(src)
    print('\n=== FOCUS source mix July/August ===')
    display(src.loc[src['report_month'].isin(FOCUS_MONTHS)])
else:
    print('column commission_monthly_source not present — skip')
